# message的使用
## 1、消息格式
举例1：json格式

In [1]:
# from Demos.dde.ddeclient import conversation   # 删除或注释
from dotenv import load_dotenv
import os
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, SystemMessage

load_dotenv(override=True)
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

model = init_chat_model(
    model="deepseek-v4-flash",
    model_provider='deepseek',
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
)



In [ ]:
# json格式
# 通过JSON初始化
messages = [
    {"role": "system", "content": "你是一个善于给出通俗易懂解释的AI助手"},
    {"role": "user", "content": "你好"},
    {"role": "assistant", "content": "你好！我能帮你什么？"},
    {"role": "user", "content": "我刚才问了什么问题"}
]
response = model.invoke(messages)
print(response.content)

## 4、 toolmessage的使用
举例1 ：jason格式

In [ ]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
# 从.env文件中加载环境变量
load_dotenv(override=True)
CLOSEAI_API_KEY = os.getenv("CLOSEAI_API_KEY")
CLOSEAI_BASE_URL = os.getenv("CLOSEAI_BASE_URL")
model = init_chat_model(
model="gpt-5.4-mini",
model_provider="openai",
api_key=CLOSEAI_API_KEY,
base_url=CLOSEAI_BASE_URL
)
def get_weather(city: str) -> str:
    return "不错哦~"
# 模拟模型绑定工具
model_with_tools = model.bind_tools([get_weather])
ai_message = {
"role": "assistant",
"content": "",
"tool_calls": [{
"name": "get_weather",
"args": {"location": "北京"},
"id": "call_00_nUD2NC9QRN5Cg1GaoIkBJQ4s"
}]
}
tool_message = {
"role": "tool",
"content": "今天北京天气晴朗，万里无云~",
    "tool_call_id": "call_00_nUD2NC9QRN5Cg1GaoIkBJQ4s"
}
messages = [
{"role": "user", "content": "北京天气如何"},
ai_message,
tool_message
]
response = model.invoke(messages)
print(response)

## 5、对话历史的优化
举例：

In [2]:
def keep_recent_messages(messages,max_pairs=3):
    """保留最近的n轮
    max_pairs:最大的保留轮数（每轮=user+assistant）"""

    #分离system消息和对话消息
    system_messages =[m for m in messages if m.get("role") == "system"]
    conversation_messages =[m for m in messages if m.get("role") != "system"]
    #只保留最近的消息对
    recent_messages = conversation_messages[-(max_pairs*2):]

    #返回系统消息个和最近消息对
    return system_messages+ recent_messages

In [3]:
# 初始化
long_conversation = [
    {"role": "system", "content": "你是 Python 导师"}
]
# 第 1 轮
long_conversation.append({"role": "user", "content": "什么是列表？用一句解释"})
r1 = model.invoke(long_conversation)
long_conversation.append({"role": "assistant", "content": r1.content})
# 第 2 轮
long_conversation.append({"role": "user", "content": "列表和元组有什么区别？用一句解释"})
r2 = model.invoke(long_conversation)
long_conversation.append({"role": "assistant", "content": r2.content})
# 第 3 轮
long_conversation.append({"role": "user", "content": "什么是字典呢？用一句解释"})
r3 = model.invoke(long_conversation)
long_conversation.append({"role": "assistant", "content": r3.content})
print(f"原始消息数: {len(long_conversation)}")
# 优化：只保留最近 2 轮
optimized = keep_recent_messages(long_conversation, max_pairs=2)
print(f"优化后消息数: {len(optimized)}")
print(f"保留的内容: system + 最近2轮对话")
# 添加新的用户问题
optimized.append({"role": "user", "content": "我第一个问题问的是什么？"})
# 使用优化后的历史
response = model.invoke(optimized)
print(f"\nAI 回复: {response.content}")

原始消息数: 7
优化后消息数: 5
保留的内容: system + 最近2轮对话

AI 回复: 您第一个问题是：“列表和元组有什么区别？用一句解释”


# 6、多轮对话聊天机器人
举例：

In [ ]:
EXIT_WORD="quit"
MAX_PAIRS_HISTORY=3
# 3维护一个消息列表
messages=[
    {
        "role": "assistant",
        "content":"你是小姑姐姐，是最牛逼的数字员工，也是一名耐心的ai助手，可以回答学上的问题"
    }
]

print(f"请输入具体的问题，当输入{EXIT_WORD}的时候，结束对话")
i=1#描述对话的轮数
while True:
    print("\n","="*10,f"第{i}对话开始","="*10,"\n")
    user_input=input("请输入：")
    #判断是否结束当前对话
    if user_input == EXIT_WORD:
        print("对话结束，欢迎下次再来")
    break

    #将用户信息添加到消息列表中
    messages.append({"role": "user", "content": user_input})
    print("小姑姐姐：",end="",flush=True)
    #拼接ai回复的消息信息
    reply_content=""
    #优化历史记忆
    memory_messages=keep_recent_messages(messages,max_pairs=MAX_PAIRS_HISTORY)
    # model.invoke(memory_messages)
    for chunk in model.stream(memory_messages):
        if chunk.content:
            print(chunk.content,end="",flush=True)
            reply_content += chunk.content

       print("\n","="*10,f"第{i}对话结束","="*10,"\n")
    i+=1
    #将模型的相应添加到消息列表中
    messages.append({"role": "assistant", "content": reply_content})
